In [15]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import json

Vocab size: 16507


In [9]:
X_train = torch.load("/kaggle/input/datasets/konstantineendeladze/nlphomework3/data/X_train.pt")
y_train = torch.load("/kaggle/input/datasets/konstantineendeladze/nlphomework3/data/y_train.pt")
X_test = torch.load("/kaggle/input/datasets/konstantineendeladze/nlphomework3/data/X_test.pt")

with open("/kaggle/input/datasets/konstantineendeladze/nlphomework3/data/vocab.json", "r") as f:
    vocab = json.load(f)

print("Vocab size:", len(vocab))

In [10]:
class SentimentDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]


BATCH_SIZE = 256  # IMPORTANT (bigger batch = faster, but safe on Kaggle GPU)

train_dataset = SentimentDataset(X_train, y_train)
test_dataset = SentimentDataset(X_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [13]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, output_dim=5):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(hidden)


In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LSTMModel(vocab_size=len(vocab))
model = model.to(device)

print(model)

/home/konstantine/Documents/work/nlp/NL-Homework-3/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:371: UserWarning: Found GPU0 NVIDIA GeForce MX130 which is of compute capability (CC) 5.0.
The following list shows the CCs this version of PyTorch was built for and the hardware CCs it supports:
- 7.5 which supports hardware CC >=7.5,<8.0
- 8.0 which supports hardware CC >=8.0,<9.0 except {8.7}
- 8.6 which supports hardware CC >=8.6,<9.0 except {8.7}
- 9.0 which supports hardware CC >=9.0,<10.0
- 10.0 which supports hardware CC >=10.0,<11.0 except {10.1}
- 12.0 which supports hardware CC >=12.0,<13.0
Please follow the instructions at https://pytorch.org/get-started/locally/ to install a PyTorch release that supports one of these CUDA versions: 12.6
  _warn_unsupported_code(d, device_cc, code_ccs)
/home/konstantine/Documents/work/nlp/NL-Homework-3/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:489: UserWarning: 
NVIDIA GeForce MX130 with CUDA capability sm_50 is not c

RuntimeError: cuDNN version 91900 is not compatible with devices with SM < 7.5. Please install a version of PyTorch with a compatible cuDNN version. https://github.com/pytorch/pytorch/blob/main/RELEASE.md#release-compatibility-matrix

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
import time

EPOCHS = 3

model.train()

for epoch in range(EPOCHS):
    start_time = time.time()

    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)

        loss.backward()

        # 🔥 stabilizes LSTM training
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        total_loss += loss.item()

        # accuracy
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)

    acc = correct / total
    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Loss: {total_loss/len(train_loader):.4f} | "
        f"Acc: {acc:.4f} | "
        f"Time: {epoch_time:.1f}s"
    )

In [ ]:
model.eval()

predictions = []

with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)

        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().numpy())

In [ ]:
import pandas as pd

test_df = pd.read_csv("/kaggle/input/datasets/konstantineendeladze/nlphomework3/data/raw/test.tsv", sep="\t")

submission = pd.DataFrame({
    "PhraseId": test_df["PhraseId"],
    "Sentiment": predictions
})

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)

In [ ]:
torch.save(model.state_dict(), "lstm_model.pt")